### Transformar Datos de "Locations"
1. Eliminar registros NULL del campo "location_id"
2. Eliminar registros duplicados "identicos"
3. Eliminar registros duplicados en base al campo "created_timestamp"
4. Convertir las columnas al "tipo de dato" correcto
5. Escribir los datos "transformados" en el schema "silver"

#### 1. Eliminar registros NULL del campo "location_id"

In [0]:
SELECT * 
FROM zenviro.bronze.v_locations
WHERE location_id IS NOT NULL;

#### 2. Eliminar registros duplicados "identicos"

In [0]:
SELECT * 
FROM zenviro.bronze.v_locations
WHERE location_id IS NOT NULL
ORDER BY location_id;

In [0]:
SELECT DISTINCT * 
FROM zenviro.bronze.v_locations
WHERE location_id IS NOT NULL
ORDER BY location_id;

In [0]:
-- Primera Forma
SELECT location_id,
        MAX(created_timestamp),
        MAX(location_name),
        MAX(address),
        MAX(city),
        MAX(country)
FROM zenviro.bronze.v_locations
WHERE location_id IS NOT NULL
GROUP BY location_id
ORDER BY location_id;

In [0]:
CREATE OR REPLACE TEMP VIEW v_locations_distinct
AS
  SELECT DISTINCT * 
  FROM zenviro.bronze.v_locations
  WHERE location_id IS NOT NULL
  ORDER BY location_id;

In [0]:
SELECT location_id,
        MAX(created_timestamp) AS max_created_timestamp
FROM v_locations_distinct
GROUP BY location_id;

#### 3. Eliminar registros duplicados en base al campo "created_timestamp"

In [0]:
-- Segunda Forma
WITH cte_max
AS
(
  SELECT location_id,
          MAX(created_timestamp) AS max_created_timestamp
  FROM v_locations_distinct
  GROUP BY location_id
)
SELECT t.*
FROM v_locations_distinct t
INNER JOIN cte_max m
ON t.location_id = m.location_id
AND t.created_timestamp = m.max_created_timestamp

#### 4. Convertir las columnas al "tipo de dato" correcto

In [0]:
WITH cte_max
AS
(
  SELECT location_id,
          MAX(created_timestamp) AS max_created_timestamp
  FROM v_locations_distinct
  GROUP BY location_id
)
SELECT CAST(t.created_timestamp AS TIMESTAMP) AS created_timestamp,
       t.location_id,
       t.location_name,
       t.address,
       t.city,
       t.country
FROM v_locations_distinct t
INNER JOIN cte_max m
ON t.location_id = m.location_id
AND t.created_timestamp = m.max_created_timestamp

#### 5. Escribir los datos "transformados" en el schema "silver"

In [0]:
CREATE OR REPLACE TABLE zenviro.silver.locations
AS
  WITH cte_max
  AS
  (
    SELECT location_id,
            MAX(created_timestamp) AS max_created_timestamp
    FROM v_locations_distinct
    GROUP BY location_id
  )
  SELECT CAST(t.created_timestamp AS TIMESTAMP) AS created_timestamp,
        t.location_id,
        t.location_name,
        t.address,
        t.city,
        t.country
  FROM v_locations_distinct t
  INNER JOIN cte_max m
  ON t.location_id = m.location_id
  AND t.created_timestamp = m.max_created_timestamp

In [0]:
SELECT * FROM zenviro.silver.locations;

In [0]:
DESCRIBE EXTENDED zenviro.silver.locations;